In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv

# 1. Reproducibility configuration

SEEDS = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]


# 2. GCN architecture

class SimpleGCN(nn.Module):
    """GCN architecture used by the pretrained baseline models."""

    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()

        self.node_norm = nn.BatchNorm1d(node_dim)

        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)

        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            )

        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h_dim in hidden_dims:
            self.convs.append(GCNConv(in_dim, h_dim))
            in_dim = h_dim

        final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(final_dim // 2, 1),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        x = self.node_norm(data.x)

        # Preserve the preprocessing used during model training.
        # edge_attr is normalized when available, although standard
        # GCNConv itself only consumes x and edge_index.
        if (
            hasattr(data, "edge_attr")
            and data.edge_attr is not None
            and hasattr(self, "edge_norm")
        ):
            _ = self.edge_norm(data.edge_attr)

        u = None
        if (
            hasattr(data, "u")
            and data.u is not None
            and hasattr(self, "global_norm")
        ):
            u = self.global_norm(data.u)

        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)

        if u is not None and hasattr(self, "global_mlp"):
            h = torch.cat([node_pool, self.global_mlp(u)], dim=1)
        else:
            h = node_pool

        return self.output_mlp(h).squeeze()


# 3. Inference data loader

def create_graph_data_loader(graph_data, batch_size=64):
    """
    Convert serialized graph dictionaries into PyG Data objects.

    Inference only requires graph features. Optional y and y_soft
    fields are retained when present but are not required.
    """
    data_list = []

    for graph_dict in graph_data:
        required = ("x", "edge_index")
        missing = [key for key in required if key not in graph_dict]
        if missing:
            raise KeyError(
                f"Graph sample is missing required field(s): {missing}"
            )

        data_kwargs = {
            "x": graph_dict["x"],
            "edge_index": graph_dict["edge_index"],
            "edge_attr": graph_dict.get("edge_attr", None),
            "u": graph_dict.get("u", None),
        }

        if "y" in graph_dict:
            data_kwargs["y"] = graph_dict["y"]

        if "y_soft" in graph_dict:
            data_kwargs["y_soft"] = graph_dict["y_soft"]

        data_list.append(Data(**data_kwargs))

    if not data_list:
        raise ValueError("No graph samples were found for prediction.")

    return DataLoader(data_list, batch_size=batch_size, shuffle=False)


# 4. Seed-specific checkpoint discovery

def extract_seed_from_filename(filename):
    """
    Extract a seed identifier from a checkpoint filename.

    Supported examples include:
        model_seed0.pt
        model_seed_42.pt
        model-seed-2025.pt
    """
    match = re.search(r"(?i)seed[_-]?(\d+)", filename)
    return int(match.group(1)) if match else None


def get_seed_model_file_paths(model_folder, seeds=SEEDS, ext=".pt"):
    """
    Return one checkpoint path for each requested seed.

    For reproducibility, a seed is accepted only when exactly one
    matching checkpoint is found. Missing or duplicated seeds are
    reported explicitly rather than silently selecting a file.
    """
    if not os.path.isdir(model_folder):
        raise FileNotFoundError(
            f"Model directory does not exist: {model_folder}"
        )

    candidates = [
        os.path.join(model_folder, filename)
        for filename in sorted(os.listdir(model_folder))
        if filename.endswith(ext)
    ]

    seed_to_paths = {seed: [] for seed in seeds}

    for model_path in candidates:
        model_seed = extract_seed_from_filename(
            os.path.basename(model_path)
        )
        if model_seed in seed_to_paths:
            seed_to_paths[model_seed].append(model_path)

    selected_paths = []
    missing_seeds = []
    duplicate_seeds = {}

    for seed in seeds:
        matches = seed_to_paths[seed]

        if len(matches) == 1:
            selected_paths.append(matches[0])
        elif len(matches) == 0:
            missing_seeds.append(seed)
        else:
            duplicate_seeds[seed] = matches

    if missing_seeds:
        raise FileNotFoundError(
            "No checkpoint was found for seed(s): "
            + ", ".join(map(str, missing_seeds))
        )

    if duplicate_seeds:
        details = "; ".join(
            f"seed {seed}: {[os.path.basename(p) for p in paths]}"
            for seed, paths in duplicate_seeds.items()
        )
        raise RuntimeError(
            "Multiple checkpoints were found for the same seed. "
            f"Keep one checkpoint per seed. {details}"
        )

    return selected_paths


# 5. Multi-model Tg prediction

def predict_with_multiple_gcn_models(
    model_paths,
    feature_data_path,
    output_path=None,
    batch_size=64,
):
    """
    Run seed-specific GCN checkpoints on one graph-feature dataset.

    Each checkpoint is independently inverse-standardized using the
    y_mean and y_std values stored in that checkpoint.
    """
    if not model_paths:
        raise ValueError("No model checkpoints were provided.")

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"Using device: {device}")

    graph_data_file_path = os.path.join(
        feature_data_path,
        "graph_data.pt",
    )
    if not os.path.isfile(graph_data_file_path):
        raise FileNotFoundError(
            f"graph_data.pt was not found: {graph_data_file_path}"
        )

    # graph_data.pt is generated by this project and is therefore
    # treated as a trusted local artifact.
    graph_data = torch.load(
        graph_data_file_path,
        map_location="cpu",
        weights_only=False,
    )

    if not graph_data:
        raise ValueError(
            f"graph_data.pt is empty: {graph_data_file_path}"
        )

    data_loader = create_graph_data_loader(
        graph_data,
        batch_size=batch_size,
    )
    all_model_predictions = {}

    for model_file_path in model_paths:
        model_basename = os.path.basename(model_file_path)
        model_seed = extract_seed_from_filename(model_basename)

        print(
            f"\nLoading checkpoint for seed {model_seed}: "
            f"{model_basename}"
        )

        checkpoint = torch.load(
            model_file_path,
            map_location=device,
            weights_only=False,
        )

        required_keys = [
            "node_dim",
            "edge_dim",
            "global_dim",
            "hidden_dims",
            "dropout",
            "model_state_dict",
            "y_mean",
            "y_std",
        ]
        missing_keys = [
            key for key in required_keys
            if key not in checkpoint
        ]
        if missing_keys:
            raise KeyError(
                f"{model_basename} is missing checkpoint field(s): "
                f"{missing_keys}"
            )

        model = SimpleGCN(
            node_dim=checkpoint["node_dim"],
            edge_dim=checkpoint["edge_dim"],
            global_dim=checkpoint["global_dim"],
            hidden_dims=checkpoint["hidden_dims"],
            dropout=checkpoint["dropout"],
        ).to(device)

        model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )
        model.eval()

        model_predictions = []

        with torch.no_grad():
            for batch in tqdm(
                data_loader,
                desc=f"Predicting seed {model_seed}",
            ):
                batch = batch.to(device)
                pred = model(batch)
                model_predictions.extend(
                    pred.detach().cpu().reshape(-1).numpy()
                )

        model_predictions = np.asarray(
            model_predictions,
            dtype=float,
        )

        # Convert predictions from the training-standardized scale
        # back to the original Tg scale.
        y_mean = float(checkpoint["y_mean"])
        y_std = float(checkpoint["y_std"])
        model_predictions = (
            model_predictions * y_std + y_mean
        )

        if len(model_predictions) != len(graph_data):
            raise RuntimeError(
                f"Prediction count mismatch for seed {model_seed}: "
                f"{len(model_predictions)} predictions for "
                f"{len(graph_data)} graphs."
            )

        all_model_predictions[
            f"seed_{model_seed}"
        ] = model_predictions

    prediction_df = pd.DataFrame(all_model_predictions)

    if output_path is None:
        output_path = os.path.join(
            feature_data_path,
            "multi_GCN_Tg_predictions.csv",
        )

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    prediction_df.to_csv(
        output_path,
        index=False,
    )

    print(
        f"\nPredictions from {len(model_paths)} seed-specific "
        f"models were saved to: {output_path}"
    )

    return prediction_df


# 6. User configuration and execution

if __name__ == "__main__":
    # Use project-relative paths so the notebook can be shared
    # without exposing machine-specific directory information.
    MODEL_DIR = os.path.join(
        "checkpoints",
        "Tg",
    )
    FEATURE_DATA_DIR = os.path.join(
        "data",
        "Tg",
        "prediction",
    )
    OUTPUT_CSV = os.path.join(
        "results",
        "Tg",
        "multi_GCN_Tg_predictions.csv",
    )

    pretrained_model_paths = get_seed_model_file_paths(
        MODEL_DIR,
        seeds=SEEDS,
    )

    predict_with_multiple_gcn_models(
        model_paths=pretrained_model_paths,
        feature_data_path=FEATURE_DATA_DIR,
        output_path=OUTPUT_CSV,
        batch_size=64,
    )
